In [1]:

!pip install -q "langchain==0.2.16" "langchain-core==0.2.43" "langchain-groq"



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 397.1/397.1 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.8/311.8 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 66.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 1.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.28.0 requires tenacity<10.0.0,>=9.0.0, but you have tenacity 8.5.0 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=

**1. Basic LLM call**

In [2]:
import os
from google.colab import userdata
from langchain_groq import ChatGroq

# Securely fetching the key from Colab Secrets
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

# Initialize the Llama-3.3-70B model
llm=ChatGroq(model="llama-3.3-70b-versatile",temperature=0.5)

try:
    response=llm.invoke("What are the 3 core principles of LangChain?")
    print(response.content)
except Exception as e:
    print(f"Connection Error: {e}")

LangChain is a framework for building applications that utilize large language models. The three core principles of LangChain are:

1. **Modularity**: Breaking down complex applications into smaller, modular components that can be easily integrated and reused.
2. **Composability**: Allowing these modular components to be combined in various ways to create more complex and powerful applications.
3. **Chainability**: Enabling these components to be chained together, with the output of one component serving as the input to the next, to create workflows and processes that can be used to solve real-world problems.

By following these principles, developers can build applications that are flexible, scalable, and can take advantage of the capabilities of large language models.


**2. PromptTemplate usage**

In [3]:
from langchain_core.prompts import PromptTemplate

prompt=PromptTemplate.from_template("Explain {topic} in simple terms")
formatted_prompt=prompt.format(topic="GenAI")

print(formatted_prompt)
print("\n--- LLM Output ---\n")
print(llm.invoke(formatted_prompt).content)

Explain GenAI in simple terms

--- LLM Output ---

GenAI, short for General Artificial Intelligence, refers to a type of artificial intelligence that can perform a wide range of tasks, similar to human intelligence. Here's a simple explanation:

**What is GenAI:**
GenAI is a computer system that can learn, reason, and apply knowledge across various domains, much like a human. It's designed to be versatile, flexible, and able to adapt to new situations, just like humans do.

**Key characteristics:**

1. **General-purpose:** GenAI can perform many tasks, from simple to complex, without being specifically programmed for each task.
2. **Learning:** GenAI can learn from experience, data, and interactions, and improve its performance over time.
3. **Reasoning:** GenAI can reason, make decisions, and solve problems using logic, intuition, and common sense.
4. **Adaptability:** GenAI can adapt to new situations, environments, and tasks, and adjust its behavior accordingly.

**Examples:**
Imagi

**3. Simple Chain**

In [6]:
from langchain_core.output_parsers import StrOutputParser

# Create the Chain
chain=prompt | llm | StrOutputParser()

# Run the Chain
result = chain.invoke({"topic": "Tokenization for Backend Developers"})
print("\n--- Chain Output ---")
print(result)


--- Chain Output ---
**Tokenization for Backend Developers**

### Introduction

Tokenization is a crucial concept in backend development, particularly when dealing with authentication and authorization. In simple terms, tokenization is the process of replacing sensitive data with a unique, non-sensitive token.

### What is Tokenization?

Tokenization is a technique where a sensitive piece of data, such as a password or credit card number, is replaced with a unique token. This token is then used to authenticate or authorize a user, without exposing the sensitive data.

### How Tokenization Works

Here's a step-by-step explanation of the tokenization process:

1. **User Authentication**: A user attempts to log in to an application or service.
2. **Token Generation**: The backend generates a unique token, which is associated with the user's sensitive data (e.g., password).
3. **Token Storage**: The token is stored on the client-side (e.g., browser) or server-side (e.g., database).
4. **T

**4. Agent with tool**

In [7]:
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage

# Model
llm=ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

# Tool
@tool
def calculate_checkout_total(quantity: int, price_per_unit: float) -> str:
    """Calculates the final checkout price including a ₹97 shipping fee."""
    shipping=97
    subtotal=quantity * price_per_unit
    grand_total=subtotal + shipping
    return f"Subtotal: ₹{subtotal}. Grand Total: ₹{grand_total} (includes ₹{shipping} shipping)."

tools=[calculate_checkout_total]
tool_map={t.name: t for t in tools}

# Bind tools to model
llm_with_tools=llm.bind_tools(tools)

# First call
messages=[HumanMessage("Calculate the total for 3 items at ₹300 each.")]
response=llm_with_tools.invoke(messages)
messages.append(response)

# Execute tool calls
for tool_call in response.tool_calls:
    result=tool_map[tool_call["name"]].invoke(tool_call["args"])
    messages.append(ToolMessage(result, tool_call_id=tool_call["id"]))

# Final answer
final=llm_with_tools.invoke(messages)
print("Final Output:", final.content)

Final Output: The total for 3 items at ₹300 each is ₹997.


**5. Memory example**

In [8]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# Prompt with a messages placeholder for history
memory_prompt=ChatPromptTemplate.from_messages([("system", "You are a helpful coding assistant."),MessagesPlaceholder(variable_name="chat_history"),("human", "{input}"),])

# Build the chain (LCEL style)
chain=memory_prompt | llm

# Simple in-memory store (session_id → history object)
store={}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in store:
        store[session_id]=InMemoryChatMessageHistory()
    return store[session_id]

# Wrap chain with history management
conversation=RunnableWithMessageHistory(chain,get_session_history,input_messages_key="input",history_messages_key="chat_history",)

# Config ties messages to a specific session
config={"configurable": {"session_id": "my_session"}}

# Turn 1
conversation.invoke(
    {"input": "Hi! I'm learning LangChain to build an NLP blog."},
    config=config
)

# Turn 2 — model remembers the previous message
response = conversation.invoke(
    {"input": "What should be my first technical chapter?"},
    config=config
)

print("\n--- Memory-Aware Response ---")
print(response.content)


--- Memory-Aware Response ---
For your first technical chapter, I would recommend covering the basics of setting up a LangChain project. This could include:

1. **Installing LangChain**: Walk through the process of installing LangChain and its dependencies, including any required libraries or frameworks.
2. **Configuring the Environment**: Discuss how to configure the environment for LangChain, including setting up API keys, model configurations, and other necessary settings.
3. **Creating a Basic Agent**: Introduce the concept of agents in LangChain and guide the reader through creating a basic agent that can interact with a language model.
4. **Making API Calls**: Show how to make API calls to the language model using LangChain, including how to send requests and handle responses.

Some possible chapter titles could be:

* "Getting Started with LangChain"
* "Setting Up Your First LangChain Project"
* "Introduction to LangChain Agents and APIs"
* "Building a Basic NLP Application wit